# Smoke & Fire Detection: Comparative Model Analysis

This notebook compares three object detection models (YOLOv11 Nano, RT-DETR,
Faster R-CNN) on a smoke/fire dataset. Section 1 characterizes the data. The
later sections evaluate each model on the test set, reporting overall metrics,
per-class precision/recall, and a breakdown of accuracy by object size.

The notebook is analysis only, but it loads the trained models, so it needs the
`training` (or `service`) dependency group on top of `analysis`:
`uv sync --group analysis --group training`.

### Environment

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml


# Make the repository root importable
def _repo_root() -> Path:
    for cand in (Path.cwd(), *Path.cwd().parents):
        if (cand / "pyproject.toml").exists() and (cand / "src").is_dir():
            return cand
    return Path.cwd()


REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

sns.set_theme(style="ticks", context="notebook")
print("Repo root:", REPO_ROOT)

## 1. Dataset Overview & Characterization

The dataset is downloaded (cached after the first run) and its YOLO-format
annotations are parsed into a DataFrame. The rest of the section covers the class
balance, the spread of target sizes, where in the frame the targets sit, and a
sample of annotated images.

### 1.1 Download & locate the dataset

In [ ]:
from scripts.download_dataset import download_dataset

DATA_ROOT = download_dataset(REPO_ROOT / "data")
print("Dataset root:", DATA_ROOT)
print("Contents:", sorted(p.name for p in DATA_ROOT.iterdir()))

### 1.2 Dataset configuration & splits

The class names come from `data.yaml`, and the train/val/test splits are found by
locating every `labels/` directory.

In [ ]:
from scripts.dataset import discover_splits

with open(DATA_ROOT / "data.yaml") as fh:
    DATA_CONF = yaml.safe_load(fh)

_names = DATA_CONF.get("names", [])
CLASS_NAMES = (
    {i: str(n) for i, n in enumerate(_names)}
    if isinstance(_names, list)
    else {int(i): str(n) for i, n in _names.items()}
)
print("Classes:", CLASS_NAMES)

SPLITS = discover_splits(DATA_ROOT)
for name, dirs in SPLITS.items():
    n_lbl = len(list(dirs["labels"].glob("*.txt")))
    print(f"{name:>8}: {n_lbl:>6} label files  (images: {bool(dirs['images'])})")

### 1.3 Load annotations into a tidy DataFrame

Each YOLO label line is `class x_center y_center width height`, normalized to
`[0, 1]`. Labels are parsed into one row per box, kept normalized, alongside a
per-image table that also retains the background images (empty label files).
Boxes that spill past the frame are clipped to `[0, 1]`. Two per-box columns are
derived: `area` (fraction of the frame covered) and `dist_center` (distance of
the box center from the frame center).

In [ ]:
from scripts.dataset import parse_yolo_labels

ann, img_inv = parse_yolo_labels(SPLITS)

ann["class_name"] = ann["class_id"].map(CLASS_NAMES).fillna(ann["class_id"].astype(str))

# Box edges from YOLO center/size; count the ones that spill over, then clip to frame
x1, x2 = ann["x_center"] - ann["width"] / 2, ann["x_center"] + ann["width"] / 2
y1, y2 = ann["y_center"] - ann["height"] / 2, ann["y_center"] + ann["height"] / 2
n_oob = int(((x1 < 0) | (x2 > 1) | (y1 < 0) | (y2 > 1)).sum())

x1, x2, y1, y2 = x1.clip(0, 1), x2.clip(0, 1), y1.clip(0, 1), y2.clip(0, 1)
ann["x_center"], ann["width"] = (x1 + x2) / 2, x2 - x1
ann["y_center"], ann["height"] = (y1 + y2) / 2, y2 - y1

ann["area"] = ann["width"] * ann["height"]
ann["dist_center"] = np.hypot(ann["x_center"] - 0.5, ann["y_center"] - 0.5)

n_bg = int((img_inv["n_objects"] == 0).sum())
print(
    f"{len(ann):,} boxes across {len(img_inv):,} images "
    f"({n_bg:,} background = {n_bg / len(img_inv):.0%}); "
    f"{n_oob:,} boxes ({n_oob / len(ann):.1%}) clipped to image bounds"
)

# Background images are spread across every split
bg_by_split = (
    img_inv.assign(background=img_inv["n_objects"] == 0)
    .groupby("split")
    .agg(images=("image", "size"), background=("background", "sum"))
)
bg_by_split["background_%"] = (
    bg_by_split["background"] / bg_by_split["images"] * 100
).round(2)

print("Background images by split:")
display(bg_by_split)
ann.head()

In [ ]:
labels_dir = SPLITS["test"]["labels"]

print("A few label examples:\n")
for name in ["AoF07082", "AoF07096", "WEB11228", "AoF07880"]:
    text = (labels_dir / f"{name}.txt").read_text()
    print(f"{name}.txt:")
    print(text or "<empty>\n")

### 1.4 Class distribution (smoke vs. fire)

Instances per class in each split. The two classes are roughly balanced overall.

In [ ]:
inst_by_class = ann.groupby(["split", "class_name"]).size().unstack(fill_value=0)

print("Instances per class per split:")
display(inst_by_class)

shares = (
    ann["class_name"]
    .value_counts(normalize=True)
    .reindex(list(CLASS_NAMES.values()))
    .mul(100)
    .round(1)
)
print("Overall class share (%):", shares.to_dict())

ax = inst_by_class.plot(kind="bar", figsize=(7, 4), width=0.8)

for container in ax.containers:
    ax.bar_label(container, fmt="{:,.0f}", padding=2, fontsize=8, color="0.3")

ax.margins(y=0.10)
ax.set(xlabel="", ylabel="instances")
ax.set_title("Instances per class, by split", loc="left")
ax.tick_params(axis="x", rotation=0)
ax.legend(title=None)

sns.despine()
plt.tight_layout()
plt.show()

### 1.5 Box size by class

The share of each class's boxes that are small, medium, or large, by area as a
fraction of the image. Fire is mostly small (64% under 1% of the frame) and smoke
mostly large (56% over 10%), so size is a good indicator of the class.

In [ ]:
size_bucket = pd.cut(
    ann["area"] * 100,
    [0, 1, 10, 100],
    labels=["small (<1%)", "medium (1-10%)", "large (>10%)"],
)
comp = (
    pd.crosstab(ann["class_name"], size_bucket, normalize="index")
    .mul(100)
    .reindex(["fire", "smoke"])
)

fig, ax = plt.subplots(figsize=(8, 2.8))
left = np.zeros(len(comp))

for color, bucket in zip(sns.color_palette("YlOrRd", 4)[:3], comp.columns, strict=True):
    ax.barh(comp.index, comp[bucket], left=left, height=0.6, color=color, label=bucket)
    for j, v in enumerate(comp[bucket]):
        ax.text(left[j] + v / 2, j, f"{v:.0f}%", ha="center", va="center")
    left += comp[bucket].to_numpy()

ax.set_xlim(0, 100)
ax.set(xlabel="share of boxes (%)", ylabel="")
ax.set_title("Box size composition by class", loc="left")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=3, frameon=False)

sns.despine()
plt.tight_layout()
plt.show()

### 1.6 Spatial distribution of box centers

Both classes cluster near the middle of the frame, slightly above center, where
smoke and fire tend to rise. The two panels look almost the same, so position
(unlike size) says little about the class.

In [ ]:
cmap = sns.color_palette("rocket_r", as_cmap=True).copy()
cmap.set_under("white")  # empty cells stay white

edges = np.linspace(0, 1, 26)
counts = {
    cls: np.histogram2d(d["x_center"], d["y_center"], bins=[edges, edges])[0]
    for cls, d in ann.groupby("class_name")
}
# cap the shared scale at the 99th percentile so a few hot center cells don't
# flatten the contrast everywhere else
vmax = np.percentile(np.concatenate([c.ravel() for c in counts.values()]), 99)

fig, axes = plt.subplots(1, 2, figsize=(8.5, 4), sharey=True, constrained_layout=True)

for ax, cls in zip(axes, ["smoke", "fire"], strict=True):
    mesh = ax.pcolormesh(edges, edges, counts[cls].T, cmap=cmap, vmin=1, vmax=vmax)
    ax.set_aspect("equal")
    ax.invert_yaxis()  # match image coordinates (y down)
    ax.set_title(cls, loc="left")
    ax.set_xlabel("x")

axes[0].set_ylabel("y")

fig.colorbar(mesh, ax=axes, shrink=0.85, label="boxes per cell")
fig.suptitle("Box centers cluster in the middle of the frame")

plt.show()

### 1.7 Sample annotated images

A small set of test images, hand-picked to show the range of the data, with their
boxes drawn (smoke yellow, fire red) and each titled with class, object count, and
largest-box size.

In [ ]:
from scripts.viz import draw_sample, find_image

CLASS_COLORS = {0: "yellow", 1: "red"}  # smoke, fire

# Hand-picked test images for the showcase
SAMPLE_IMAGES = [
    "AoF07806",
    "PublicDataset01095",
    "WEB10441",
    "AoF08246",
    "PublicDataset01075",
    "WEB10977",
]

test = ann[ann["split"] == "test"]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, image in zip(axes.ravel(), SAMPLE_IMAGES, strict=False):
    boxes = test[test["image"] == image]
    names = set(boxes["class_name"])
    cls = "fire+smoke" if len(names) == 2 else next(iter(names), "?")
    title = (
        f"{cls} · {len(boxes)} obj · {boxes['area'].max() * 100:.0f}% of frame"
        if len(boxes)
        else image
    )
    draw_sample(
        ax,
        find_image(SPLITS["test"]["images"], image),
        boxes,
        class_colors=CLASS_COLORS,
        title=title,
    )

fig.suptitle("smoke = yellow, fire = red", y=1.0)

plt.tight_layout()
plt.show()

### 1.8 Summary

In [ ]:
scene = img_inv[img_inv["n_objects"] > 0]  # n_bg computed in section 1.3

class_share = (
    ann["class_name"]
    .value_counts(normalize=True)
    .reindex(list(CLASS_NAMES.values()))
    .mul(100)
)

# Size skew (see section 1.5): fire mostly small, smoke mostly large.
area_pct = ann["area"] * 100
fire_small = (area_pct[ann["class_name"] == "fire"] < 1).mean() * 100
smoke_large = (area_pct[ann["class_name"] == "smoke"] > 10).mean() * 100

summary = pd.DataFrame(
    {
        "metric": [
            "total images",
            "background images (all splits)",
            "annotations (boxes)",
            f"class share % ({' / '.join(CLASS_NAMES.values())})",
            "objects per scene (median)",
            "fire boxes small (<1% of frame)",
            "smoke boxes large (>10% of frame)",
        ],
        "value": [
            f"{len(img_inv):,}",
            f"{n_bg:,} ({n_bg / len(img_inv):.0%})",
            f"{len(ann):,}",
            " / ".join(f"{v:.0f}" for v in class_share),
            int(scene["n_objects"].median()),
            f"{fire_small:.0f}%",
            f"{smoke_large:.0f}%",
        ],
    }
)

display(summary)

The dataset is varied but has clear regularities: fire and smoke are roughly
balanced, most scenes hold a single object, box centers cluster toward the middle
of the frame (slightly above it), and about 46% of images are object-free
backgrounds. Box size is an indicator of class membership (fire is small, smoke is large).

## 2. Model Loading & Setup

The three fine-tuned models load from `weights/`: YOLOv11 Nano and RT-DETR via
Ultralytics, Faster R-CNN by rebuilding the torchvision architecture and loading
its `state_dict`. The table reports parameter count and on-disk size.

In [ ]:
import time

import torch
from ultralytics import RTDETR, YOLO

WEIGHTS = REPO_ROOT / "weights"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}  device={DEVICE}")

In [ ]:
from src.models.fasterrcnn import LABEL_OFFSET, build_model

In [ ]:
yolo = YOLO(str(WEIGHTS / "yolo11n_best.pt"))
rtdetr = RTDETR(str(WEIGHTS / "rtdetr_best.pt"))

frcnn = build_model(pretrained=False)
frcnn.load_state_dict(torch.load(WEIGHTS / "fasterrcnn_best.pt", map_location=DEVICE))
frcnn = frcnn.to(DEVICE).eval()

print("loaded yolo11n, rtdetr, fasterrcnn")

In [ ]:
def efficiency(model, weights_file):
    """Parameter count (millions) and on-disk file size (MB) for one model."""
    # FLOPs omitted: input-size dependent, and not comparable across these models.
    return {
        "parameters_M": sum(p.numel() for p in model.parameters()) / 1e6,
        "size_mb": (WEIGHTS / weights_file).stat().st_size / 1e6,
    }


model_info = pd.DataFrame.from_dict(
    {
        "YOLOv11 Nano": efficiency(yolo.model, "yolo11n_best.pt"),
        "RT-DETR": efficiency(rtdetr.model, "rtdetr_best.pt"),
        "Faster R-CNN": efficiency(frcnn, "fasterrcnn_best.pt"),
    },
    orient="index",
).round(1)
display(model_info)

## 3. Evaluation

Score all three on the test set through one pipeline so the numbers stay
comparable: mAP (overall and per class), precision / recall per class with P-R
curves, AP by object size (small / medium / large) for the robustness angle, and
inference latency. Collect everything in a single comparison table.

## 4. Conclusions & Recommendations

Synthesize the results, justify YOLOv11 Nano as the deployed default (speed and
size against accuracy), note where the heavier models pull ahead (e.g. on small
objects), and give the deployment implications.